# Week 7 Assignment
# Document Question Answering System using RAG (Retrieval-Augmented Generation)

## Name: Tejender Singh

### Objective
To build a Retrieval-Augmented Generation (RAG) system that answers questions
from a custom document by combining information retrieval with a language model.

### Workflow
1. Load a PDF document
2. Split the text into chunks
3. Generate embeddings for each chunk
4. Store the embeddings in a FAISS vector database
5. Retrieve the most relevant chunks for a given question
6. Generate an answer using a local FLAN-T5 language model


## Step 1: Install Required Libraries

In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q transformers torch accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00


In [2]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


/tmp/ipykernel_1698/2244263054.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 2: Upload the Document





In [3]:
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print("Uploaded file:", pdf_filename)


Saving JULY_RESUME.pdf to JULY_RESUME.pdf
Uploaded file: JULY_RESUME.pdf


In [4]:
loader = PyPDFLoader(pdf_filename)
documents = loader.load()

print("Total Pages Loaded:", len(documents))


Total Pages Loaded: 1


## Step 3: Text Chunking




In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))


Total Chunks: 10


In [6]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}\n")
    print(chunk.page_content)
    print("-"*80)



Chunk 1

Tejender Singh
+91-9680690592|worktejender29@gmail.com|LinkedIn|GitHub
Career Objective
Aspiring Software Engineer with a foundation in Data Structures Algorithms, Java, and AI/ML, along with hands-on experience in
Data Science and software development. Passionate about building intelligent solutions to real-world problems and eager to
contribute, learn, and grow in a collaborative professional environment.
Education
Poornima College Of EngineeringJaipur, Rajasthan
--------------------------------------------------------------------------------

Chunk 2

Poornima College Of EngineeringJaipur, Rajasthan
B.Tech in Artificial Intelligence and Data Science — CGPA: 8.7 June 2023 – June 2027
D.S. Science Academy Gangapur City, Rajasthan
Higher Secondary Education — Percentage: 75% 2022
Work Experience
Celebal T echnologies Pvt. Ltd.
Data Science Intern May 18, 2026 – July 18, 2026
–Underwent intensive hands-on training in the complete ML/DL stack — classical machine learning, CNNs,

## Step 4: Create Embeddings




In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully


## Step 5: Create a FAISS Vector Database



In [8]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS Vector Store Created Successfully")


FAISS Vector Store Created Successfully


## Step 6: Load the Local Language Model (FLAN-T5)

FLAN-T5 is a free, locally-run model used to generate the final answer from
the retrieved context. No API key is required.


In [9]:
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Local Language Model Loaded Successfully")


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Local Language Model Loaded Successfully


## Step 7: Retrieval and Answer Generation Function




In [10]:
def ask_question(query, k=3):
    retriever = vector_db.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)

    print("\nRetrieved Context:\n")
    print("=" * 100)

    context = ""
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n")
        print(doc.page_content)
        print("-" * 100)
        context += doc.page_content + "\n"

    prompt = f"""Answer the question only using the context given below.
If the answer is not present in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer:"""

    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = flan_model.generate(**inputs, max_new_tokens=200)
    answer = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\n" + "=" * 100)
    print("FINAL ANSWER:", answer)
    print("=" * 100)

    return answer


## Step 8: Test Questions




In [17]:
ask_question("What is this document about?")


Retrieved Context:


Chunk 1:

Coursework: Data Structures & Algorithms, Machine Learning, DBMS, OOPs, Networking, Operating Systems
Certifications
Data Structures and Algorithms — Apna College
Programming in Java (Gold + Elite) — NPTEL
Build Real World AI Applications with Gemini & Imagen — Google Cloud Arcade
Extracurricular & Achievements
T ek-Connect(Oct 2024) — Developed a web-based platform enabling users to browse and purchase hardware products online|
Jaipur
----------------------------------------------------------------------------------------------------

Chunk 2:

Poornima College Of EngineeringJaipur, Rajasthan
B.Tech in Artificial Intelligence and Data Science — CGPA: 8.7 June 2023 – June 2027
D.S. Science Academy Gangapur City, Rajasthan
Higher Secondary Education — Percentage: 75% 2022
Work Experience
Celebal T echnologies Pvt. Ltd.
Data Science Intern May 18, 2026 – July 18, 2026
–Underwent intensive hands-on training in the complete ML/DL stack — classical machine le

'Science/Tech'

In [18]:
ask_question("What are the key topics/skills/points mentioned?")


Retrieved Context:


Chunk 1:

Coursework: Data Structures & Algorithms, Machine Learning, DBMS, OOPs, Networking, Operating Systems
Certifications
Data Structures and Algorithms — Apna College
Programming in Java (Gold + Elite) — NPTEL
Build Real World AI Applications with Gemini & Imagen — Google Cloud Arcade
Extracurricular & Achievements
T ek-Connect(Oct 2024) — Developed a web-based platform enabling users to browse and purchase hardware products online|
Jaipur
----------------------------------------------------------------------------------------------------

Chunk 2:

–Designed a multi-role Streamlit app (teacher + student portals) with session-based routing, QR-code class enrollment, and
deep-link auto-join via URL query parameters
–Integrated Supabase (PostgreSQL + Auth) as the cloud backend to store face/voice encodings, attendance logs, and confidence
scores with secure bcrypt authentication
–Deployed a Flask landing page on Vercel alongside the Streamlit app, demonstratin

'Data Structures & Algorithms, Machine Learning, DBMS, OOPs, Networking, Operating Systems Certifications Data Structures and Algorithms — Apna College Programming in Java (Gold + Elite) — NPTEL Build Real World AI Applications with Gemini & Imagen — Google Cloud Arcade Extracurricular & Achievements T ek-Connect(Oct 2024) — Developed a web-based platform enabling users to browse and purchase hardware products online| Jaipur –Designed a multi-role Streamlit app (teacher + student portals) with session-based routing, QR-code class enrollment, and deep-link auto-join via URL query parameters Integrated Supabase (PostgreSQL + Auth) as the cloud backend to store face/voice encodings, attendance logs, and confidence'

In [19]:
ask_question("What is the educational qualification?")


Retrieved Context:


Chunk 1:

Poornima College Of EngineeringJaipur, Rajasthan
B.Tech in Artificial Intelligence and Data Science — CGPA: 8.7 June 2023 – June 2027
D.S. Science Academy Gangapur City, Rajasthan
Higher Secondary Education — Percentage: 75% 2022
Work Experience
Celebal T echnologies Pvt. Ltd.
Data Science Intern May 18, 2026 – July 18, 2026
–Underwent intensive hands-on training in the complete ML/DL stack — classical machine learning, CNNs, RNNs,
----------------------------------------------------------------------------------------------------

Chunk 2:

Coursework: Data Structures & Algorithms, Machine Learning, DBMS, OOPs, Networking, Operating Systems
Certifications
Data Structures and Algorithms — Apna College
Programming in Java (Gold + Elite) — NPTEL
Build Real World AI Applications with Gemini & Imagen — Google Cloud Arcade
Extracurricular & Achievements
T ek-Connect(Oct 2024) — Developed a web-based platform enabling users to browse and purchase hardware prod

'B.Tech in Artificial Intelligence and Data Science'

## Step 9: System Metrics Report

In [15]:
print("===== SYSTEM METRICS =====")
print("Total Pages:", len(documents))
print("Total Chunks:", len(chunks))
print("Chunk Size:", 500)
print("Chunk Overlap:", 50)
print("Embedding Model: all-MiniLM-L6-v2")
print("Embedding Dimension: 384")
print("Vector Database: FAISS")
print("Retriever Top-K:", 3)
print("LLM: google/flan-t5-base (local)")


===== SYSTEM METRICS =====
Total Pages: 1
Total Chunks: 10
Chunk Size: 500
Chunk Overlap: 50
Embedding Model: all-MiniLM-L6-v2
Embedding Dimension: 384
Vector Database: FAISS
Retriever Top-K: 3
LLM: google/flan-t5-base (local)


## Step 10: Experiment — Comparing Chunk Sizes (Optimization Task)




In [16]:
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

small_chunks = small_splitter.split_documents(documents)
small_vector_db = FAISS.from_documents(small_chunks, embeddings)

print("Chunk size 500 -> Total Chunks:", len(chunks))
print("Chunk size 200 -> Total Chunks:", len(small_chunks))

test_query = "What are the key topics/skills/points mentioned?"

print("\n--- Retrieved with chunk_size=500 ---")
for doc in vector_db.as_retriever(search_kwargs={"k": 3}).invoke(test_query):
    print(doc.page_content[:150], "...\n")

print("\n--- Retrieved with chunk_size=200 ---")
for doc in small_vector_db.as_retriever(search_kwargs={"k": 3}).invoke(test_query):
    print(doc.page_content[:150], "...\n")


Chunk size 500 -> Total Chunks: 10
Chunk size 200 -> Total Chunks: 28

--- Retrieved with chunk_size=500 ---
Coursework: Data Structures & Algorithms, Machine Learning, DBMS, OOPs, Networking, Operating Systems
Certifications
Data Structures and Algorithms —  ...

–Designed a multi-role Streamlit app (teacher + student portals) with session-based routing, QR-code class enrollment, and
deep-link auto-join via URL ...

Tejender Singh
+91-9680690592|worktejender29@gmail.com|LinkedIn|GitHub
Career Objective
Aspiring Software Engineer with a foundation in Data Structure ...


--- Retrieved with chunk_size=200 ---
output, with applications in photo filters, animation, and game graphics
Technical Skills
Languages: Python, Java, HTML, CSS, SQL(MySQL ,PostgreSQL)
T ...

Coursework: Data Structures & Algorithms, Machine Learning, DBMS, OOPs, Networking, Operating Systems
Certifications
Data Structures and Algorithms —  ...

Career Objective
Aspiring Software Engineer with a foundation in Data St

# Experiment and Observations

1. A chunk size of 500 with an overlap of 50 provided relevant context.
2. Smaller chunks (200) improved precision but occasionally lost context.
3. FAISS enabled fast similarity search across all chunks.
4. The RAG system generated grounded answers based on retrieved chunks.
5. Using a custom document allowed question-answering over private data.


# Conclusion

This project successfully implemented the retrieval-augmented generation (RAG)
system for document question & answering.

The system loads a custom PDF document, converts it into embedding, stores
them in a FAISS vector database, retrieves the most relevant chunks for a
given question, and generates a context-aware answer using a local FLAN-T5
 model.

This approach improves factual accuracy and reduces hallucination, since the
model is required to answer using only the retrieved context, and enables
question-answering over private or custom documents.
